# Risk Scoring System — Phase 2: Build User Behavioral Profiles

**Goal:** For each card, compute a behavioral baseline — its "normal" — from that card's **legitimate transactions only**.

**Why legit-only:** If we built the baseline using fraudulent transactions too, then tested on those same transactions, the profile would already be contaminated by the fraud we're trying to detect. Results would be falsely good. Building the baseline from legit history mirrors how a real system works: learn the user's established normal, then judge new activity against it.

**Honest scope reminder — features that earned their place in Phase 1:**
- Amount (strong: fraud 8x larger)
- Hour (strong: 84% of fraud at 22:00–03:00)
- Velocity (moderate: fraud days 5 vs 3 txns)
- Category (moderate: online categories ~11x riskier)
- Merchant novelty (behavioral, per-user)
- Location novelty (behavioral, per-user)

Dropped with proof: geo-distance (identical distributions), age (45.5 vs 48.3), balance (not in data).

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import json

TRAIN = r"D:\Nithilan\Internship\QPay Internship\risk_scoring\data\fraudTrain.csv"
OUT_PROFILES = r"D:\Nithilan\Internship\QPay Internship\risk_scoring\data\user_profiles.json"
OUT_CATEGORY = r"D:\Nithilan\Internship\QPay Internship\risk_scoring\data\category_risk.json"

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)

## 1. Load & Prepare

In [ ]:
df = pd.read_csv(TRAIN, index_col=0)
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['hour'] = df['trans_date_trans_time'].dt.hour
df['date'] = df['trans_date_trans_time'].dt.date

# strip the 'fraud_' generation artifact from merchant names
df['merchant'] = df['merchant'].str.replace('^fraud_', '', regex=True)

print(f"Loaded {len(df):,} transactions, {df['cc_num'].nunique()} cards")
print(f"Fraud rate: {df['is_fraud'].mean()*100:.3f}%")

## 2. Category Base Risk (population-level, computed from full data)

Category risk is a population signal, not a per-user one. We compute each category's historical fraud rate. This becomes a lookup used at scoring time.

**Honesty note:** This uses the fraud label to compute category risk. That is legitimate here because category risk is a *known prior* a real system would maintain from historical fraud data — it is not leaking a specific transaction's label into its own score.

In [ ]:
cat_risk = df.groupby('category')['is_fraud'].mean().to_dict()
# normalise to 0-100 within observed range for use as a sub-score
cat_series = pd.Series(cat_risk)
cat_min, cat_max = cat_series.min(), cat_series.max()
cat_risk_norm = {k: round(float((v - cat_min)/(cat_max - cat_min) * 100), 2)
                 for k, v in cat_risk.items()}

print("Category risk (normalised 0-100):")
for k, v in sorted(cat_risk_norm.items(), key=lambda x:-x[1]):
    print(f"  {k:18} {v:6.2f}   (raw fraud rate {cat_risk[k]*100:.3f}%)")

with open(OUT_CATEGORY, 'w') as f:
    json.dump(cat_risk_norm, f, indent=2)
print(f"\nSaved -> {OUT_CATEGORY}")

## 3. Build Per-Card Profiles — from LEGITIMATE transactions only

This is the core of Phase 2. For each card we compute its normal behaviour using only `is_fraud == 0` rows.

In [ ]:
legit = df[df['is_fraud'] == 0].copy()

profiles = {}

for cc_num, g in legit.groupby('cc_num'):
    # daily transaction counts for this card (velocity baseline)
    daily_counts = g.groupby('date').size()

    profiles[str(cc_num)] = {
        'amt_mean'        : round(float(g['amt'].mean()), 2),
        'amt_std'         : round(float(g['amt'].std(ddof=0)), 2),
        'amt_max'         : round(float(g['amt'].max()), 2),
        'amt_median'      : round(float(g['amt'].median()), 2),
        'typical_hours'   : sorted(g['hour'].unique().tolist()),
        'known_merchants' : g['merchant'].unique().tolist(),
        'known_locations' : g['city'].unique().tolist(),
        'avg_daily_txns'  : round(float(daily_counts.mean()), 2),
        'std_daily_txns'  : round(float(daily_counts.std(ddof=0)) if len(daily_counts) > 1 else 0.0, 2),
        'max_daily_txns'  : int(daily_counts.max()),
        'txn_count'       : int(len(g)),
    }

print(f"Built profiles for {len(profiles)} cards")

# show one example profile (trimmed)
example_cc = list(profiles.keys())[0]
ex = dict(profiles[example_cc])
ex['known_merchants'] = f"[{len(ex['known_merchants'])} merchants]"
ex['known_locations'] = f"[{len(ex['known_locations'])} locations]"
ex['typical_hours']   = f"{ex['typical_hours']}"
print(f"\nExample profile (card {example_cc}):")
for k, v in ex.items():
    print(f"  {k:18}: {v}")

## 4. Sanity Checks on Profiles\nVerify the profiles are sensible before we score anything against them.

In [ ]:
amt_means = [p['amt_mean'] for p in profiles.values()]
hours_len = [len(p['typical_hours']) for p in profiles.values()]
merch_len = [len(p['known_merchants']) for p in profiles.values()]
loc_len   = [len(p['known_locations']) for p in profiles.values()]
counts    = [p['txn_count'] for p in profiles.values()]

print("Across all card profiles:")
print(f"  amt_mean      — min {min(amt_means):.2f}, median {np.median(amt_means):.2f}, max {max(amt_means):.2f}")
print(f"  typical_hours — min {min(hours_len)}, median {np.median(hours_len):.0f}, max {max(hours_len)} (out of 24)")
print(f"  known_merch   — min {min(merch_len)}, median {np.median(merch_len):.0f}, max {max(merch_len)}")
print(f"  known_locs    — min {min(loc_len)}, median {np.median(loc_len):.0f}, max {max(loc_len)}")
print(f"  txn_count     — min {min(counts)}, median {np.median(counts):.0f}, max {max(counts)}")

# honest flag: any card with too little history to trust?
thin = [c for c,p in profiles.items() if p['txn_count'] < 20]
print(f"\nCards with < 20 legit txns (thin profiles): {len(thin)}")
if thin:
    print("  These profiles are low-confidence; note this when scoring.")

## 5. A Real Concern to Check — do 'known locations' even vary per card?

If every card only ever transacts in ONE city, then 'new location' can never trigger and the location feature is dead on arrival. We check honestly.

In [ ]:
single_loc = sum(1 for p in profiles.values() if len(p['known_locations']) == 1)
single_merch = sum(1 for p in profiles.values() if len(p['known_merchants']) == 1)

print(f"Cards transacting in only ONE city : {single_loc} / {len(profiles)}")
print(f"Cards using only ONE merchant      : {single_merch} / {len(profiles)}")
print()
if single_loc > len(profiles) * 0.5:
    print("WARNING: most cards have a single location — location novelty may be weak.")
else:
    print("Locations vary per card — location novelty is viable.")
print("(We'll let the Phase 4 fraud-label evaluation make the final call on these.)")

## 6. Save Profiles

In [ ]:
with open(OUT_PROFILES, 'w') as f:
    json.dump(profiles, f)

import os
size_mb = os.path.getsize(OUT_PROFILES) / 1e6
print(f"Saved {len(profiles)} profiles -> {OUT_PROFILES} ({size_mb:.1f} MB)")

## 7. Phase 2 Summary

Fill in after running, from real outputs:

| Item | Result |
|---|---|
| Profiles built | ? |
| Built from | legitimate transactions only |
| Cards with thin history (<20 txns) | ? |
| Cards with single location | ? |
| Cards with single merchant | ? |
| Category risk lookup saved | yes/no |

**Next — Phase 3:** write the six scoring functions, each returning 0–100, using these profiles. Then Phase 4 validates the combined score against `is_fraud`.